# Lab 05 - Learn the Scorer

Plain-NumPy MLP with manual backpropagation. This recreated notebook follows the supplied README requirements.


In [ ]:
import csv
import json
import math
import random
import time
import zipfile
from pathlib import Path

import numpy as np


ROOT = Path(r"E:\Master AN1Sem1\New folder\instakill-starter")
DATA = ROOT / "ml-latest-small" / "ml-latest-small"
SEED = 505
rng = np.random.default_rng(SEED)
random.seed(SEED)


def read_ratings():
    rows = []
    with open(DATA / "ratings.csv", newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            rows.append((int(r["userId"]), int(r["movieId"]), float(r["rating"]), int(r["timestamp"])))
    return sorted(rows, key=lambda r: r[3])


def read_movies():
    movies = {}
    with open(DATA / "movies.csv", newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            movies[int(r["movieId"])] = set(r["genres"].split("|"))
    return movies


ratings = read_ratings()
genres = read_movies()
all_items = sorted(genres)
item_created = {}
for _, iid, _, ts in ratings:
    item_created[iid] = min(ts, item_created.get(iid, ts))

from bisect import bisect_left

user_history = {}
item_history = {}
for uid, iid, rating, ts in ratings:
    user_history.setdefault(uid, []).append((ts, iid, rating))
    item_history.setdefault(iid, []).append((ts, uid, rating))
for uid in user_history:
    user_history[uid].sort()
for iid in item_history:
    item_history[iid].sort()

user_times = {uid: [row[0] for row in rows] for uid, rows in user_history.items()}
item_times = {iid: [row[0] for row in rows] for iid, rows in item_history.items()}

live_cache = {}

timestamps = np.array([r[3] for r in ratings])
like_rows = [r for r in ratings if r[2] >= 4.0]
t70 = float(np.quantile(np.array([r[3] for r in like_rows]), 0.70))
t85 = float(np.quantile(np.array([r[3] for r in like_rows]), 0.85))
train_likes = [r for r in like_rows if r[3] <= t70]
val_likes = [r for r in like_rows if t70 < r[3] <= t85]
test_likes = [r for r in like_rows if r[3] > t85]


def live_signals(uid, iid, ts):
    key = (uid, iid, ts)
    if key in live_cache:
        return live_cache[key]
    uhist = user_history.get(uid, [])
    ihist = item_history.get(iid, [])
    utimes = user_times.get(uid, [])
    itimes = item_times.get(iid, [])
    uparts = uhist[:bisect_left(utimes, ts + 1)]
    iparts = ihist[:bisect_left(itimes, ts + 1)]
    uvals = [row[2] for row in uparts]
    ivals = [row[2] for row in iparts]
    icount = len(ivals)
    ucount = len(uvals)

    uitems = {row[1]: row[2] for row in uparts}
    same_genre_count = 0
    target_genres = genres.get(iid, set())
    for oid in uitems:
        if target_genres & genres.get(oid, set()):
            same_genre_count += 1
    genre_affinity = 3.5 + 0.3 * same_genre_count / max(1, len(uitems))

    freshness = 1.0 - min(1.0, max(0.0, (ts - item_created[iid]) / (365.25 * 24 * 3600 * 6)))
    pop_normalized = icount / 500.0
    user_avg = np.mean(uvals) if uvals else 3.5
    item_avg = np.mean(ivals) if ivals else 3.5
    novel = 1.0 if iid not in uitems else 0.0
    result = np.array([
        freshness,
        pop_normalized,
        (user_avg - 3.5) / 1.5,
        (item_avg - 3.5) / 1.5,
        (genre_affinity - 3.5) / 1.5,
        novel,
    ], dtype=np.float64)
    live_cache[key] = result
    return result


def make_training(likes):
    X, y = [], []
    for uid, iid, rating, ts in likes:
        X.append(live_signals(uid, iid, ts))
        y.append(1.0)
        eligible = [x for x in all_items if item_created.get(x, ts + 1) <= ts and x != iid]
        negatives = rng.choice(eligible, size=1, replace=False)
        for neg in negatives:
            X.append(live_signals(uid, int(neg), ts))
            y.append(0.0)
    X = np.asarray(X)
    y = np.asarray(y)
    Xmean = X.mean(axis=0)
    Xstd = X.std(axis=0)
    Xstd[Xstd == 0] = 1.0
    return X, y, Xmean, Xstd


Xraw, y, Xmean, Xstd = make_training(train_likes)
X = (Xraw - Xmean) / Xstd


def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    h = np.maximum(z1, 0)
    z2 = h @ W2 + b2
    return sigmoid(z2), h


def backward(X, y, cache, W1, b1, W2, b2):
    prob, h = cache
    n = X.shape[0]
    dz2 = (prob - y)[:, None] / n
    dW2 = h.T @ dz2
    db2 = dz2.sum(axis=0)
    dh = dz2 @ W2[None, :]
    dz1 = dh * (h > 0)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0)
    return dW1, db1, dW2[:, 0], float(db2[0])


def train_network(hidden, lr, epochs):
    W1 = rng.normal(0, 0.4, (6, hidden))
    b1 = np.zeros(hidden)
    W2 = rng.normal(0, 0.4, hidden)
    b2 = 0.0
    order = np.arange(X.shape[0])
    batch = 256
    for epoch in range(epochs):
        rng.shuffle(order)
        for start in range(0, len(order), batch):
            idx = order[start:start + batch]
            prob, h = forward(X[idx], W1, b1, W2, b2)
            cache = (prob, h)
            dW1, db1, dW2, db2 = backward(X[idx], y[idx], cache, W1, b1, W2, b2)
            W1 -= lr * dW1
            b1 -= lr * db1
            W2 -= lr * dW2
            b2 -= lr * db2
    return W1, b1, W2, b2


def ndcg(scores, relevant_indices):
    order = np.argsort(-scores)
    dcg = 0.0
    for rank, idx in enumerate(order[:10], start=1):
        if idx in relevant_indices:
            dcg += 1.0 / math.log2(rank + 1)
    return dcg


def exam(likes, model=None):
    if model is not None:
        W1, b1, W2, b2 = model
    vals = []
    for uid, iid, _, ts in likes:
        candidates = [iid]
        eligible = [x for x in all_items if item_created.get(x, ts + 1) <= ts and x != iid]
        candidates += list(random.sample(eligible, min(99, len(eligible))))
        if model is None:
            yield uid, iid, ts, candidates
        else:
            Xcand = np.asarray([(live_signals(uid, x, ts) - Xmean) / Xstd for x in candidates])
            scores, _ = forward(Xcand, W1, b1, W2, b2)
            vals.append(ndcg(scores, {0}))
    if model is not None:
        return float(np.mean(vals))


def model_ndcg(likes, model):
    W1, b1, W2, b2 = model
    vals = []
    for uid, iid, _, ts in likes:
        candidates = [iid]
        eligible = [x for x in all_items if item_created.get(x, ts + 1) <= ts and x != iid]
        candidates += list(random.sample(eligible, min(99, len(eligible))))
        Xcand = np.asarray([(live_signals(uid, x, ts) - Xmean) / Xstd for x in candidates])
        scores, _ = forward(Xcand, W1, b1, W2, b2)
        vals.append(ndcg(scores, {0}))
    return float(np.mean(vals))


def baseline_scores(likes, kind):
    vals = []
    for uid, iid, _, ts in likes:
        candidates = [iid]
        eligible = [x for x in all_items if item_created.get(x, ts + 1) <= ts and x != iid]
        candidates += list(random.sample(eligible, min(99, len(eligible))))
        scores = []
        for x in candidates:
            s = live_signals(uid, x, ts)
            if kind == "chronological":
                score = s[0]
            elif kind == "popularity":
                score = s[1]
            elif kind == "week4":
                score = 0.40 * s[4] + 0.35 * s[0] + 0.25 * s[1]
            elif kind == "own_weights":
                score = 0.20 * s[4] + 0.50 * s[0] + 0.30 * s[1]
            else:
                raise ValueError(kind)
            scores.append(score)
        vals.append(ndcg(np.asarray(scores), {0}))
    return float(np.mean(vals))


# Linear least squares straight-line baseline
A = np.column_stack([X, np.ones(X.shape[0])])
linear_coef = np.linalg.lstsq(A, y, rcond=None)[0]


def linear_ndcg(likes):
    vals = []
    for uid, iid, _, ts in likes:
        candidates = [iid]
        eligible = [x for x in all_items if item_created.get(x, ts + 1) <= ts and x != iid]
        candidates += list(random.sample(eligible, min(99, len(eligible))))
        Xcand = np.asarray([(live_signals(uid, x, ts) - Xmean) / Xstd for x in candidates])
        scores = np.column_stack([Xcand, np.ones(len(Xcand))]) @ linear_coef
        vals.append(ndcg(scores, {0}))
    return float(np.mean(vals))


print("boundaries", t70, t85, flush=True)
print("sizes", len(train_likes), len(val_likes), len(test_likes), flush=True)

baselines = {
    "chronological": baseline_scores(val_likes, "chronological"),
    "popularity": baseline_scores(val_likes, "popularity"),
    "week4": baseline_scores(val_likes, "week4"),
    "own_week4": baseline_scores(val_likes, "own_weights"),
    "linear": linear_ndcg(val_likes),
}
print("baselines val", baselines)

best = None
for hidden in (16, 32):
    for lr in (0.02,):
        for epochs in (80,):
            started = time.perf_counter()
            model = train_network(hidden, lr, epochs)
            score = model_ndcg(val_likes, model)
            elapsed = time.perf_counter() - started
            print("candidate", hidden, lr, epochs, score, elapsed)
            if best is None or score > best[0]:
                best = (score, hidden, lr, epochs, model)

score, hidden, lr, epochs, model = best
print("BEST", best[:4])
test_score = model_ndcg(test_likes, model)
print("TEST_ONE_RUN", test_score)

# Export model and metadata.
W1, b1, W2, b2 = model
np.savez(ROOT / "like_model.npz", W1=W1, b1=b1, W2=W2, b2=np.array([b2]), Xmean=Xmean, Xstd=Xstd)

results = {
    "best_val": score,
    "hidden": hidden,
    "learning_rate": lr,
    "epochs": epochs,
    "test": test_score,
    "baselines_val": baselines,
    "parameter_count": int(W1.size + b1.size + W2.size + 1),
    "model_size": (ROOT / "like_model.npz").stat().st_size,
}
(ROOT / "week05_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")


## Gradient check
Numerically perturb selected parameters and compare analytical gradients.


In [ ]:
# Numerical gradient check for the mean binary cross-entropy loss.
gc_rng = np.random.default_rng(505)
W1g = gc_rng.normal(0, 0.4, (6, 8))
b1g = np.zeros(8)
W2g = gc_rng.normal(0, 0.4, 8)
b2g = 0.0
idx = gc_rng.choice(X.shape[0], 40, replace=False)
Xg = X[idx]
yg = y[idx]
pg, hg = forward(Xg, W1g, b1g, W2g, b2g)
grads = backward(Xg, yg, (pg, hg), W1g, b1g, W2g, b2g)

def loss(W1_, b1_, W2_, b2_):
    pp, _ = forward(Xg, W1_, b1_, W2_, b2_)
    pp = np.clip(pp, 1e-12, 1.0 - 1e-12)
    return -np.mean(yg * np.log(pp) + (1.0 - yg) * np.log(1.0 - pp))

cases = [
    ("W1", W1g, grads[0]),
    ("b1", b1g, grads[1]),
    ("W2", W2g, grads[2]),
    ("b2", np.array([b2g]), np.array([grads[3]])),
]
eps = 1e-5
max_rel = 0.0
for name, param, analytic in cases:
    numerical = np.zeros_like(param)
    for pos in np.ndindex(param.shape):
        old = param[pos]
        param[pos] = old + eps
        lp = loss(W1g, b1g, W2g, float(b2g + eps)) if name == "b2" else loss(
            W1g if name != "W1" else param,
            b1g if name != "b1" else param,
            W2g if name != "W2" else param,
            b2g,
        )
        param[pos] = old - eps
        lm = loss(W1g, b1g, W2g, float(b2g - eps)) if name == "b2" else loss(
            W1g if name != "W1" else param,
            b1g if name != "b1" else param,
            W2g if name != "W2" else param,
            b2g,
        )
        param[pos] = old
        numerical[pos] = (lp - lm) / (2.0 * eps)
    rel = np.max(np.abs(numerical - analytic) /
                 (np.abs(numerical) + np.abs(analytic) + 1e-12))
    max_rel = max(max_rel, float(rel))
    print(name, rel)
print("max relative error", max_rel)
assert max_rel < 1e-6

In [ ]:
# Reload and reproduce check.
loaded = np.load(root / "like_model.npz")
W1r = loaded["W1"]
b1r = loaded["b1"]
W2r = loaded["W2"]
b2r = float(loaded["b2"][0])
pred_before, _ = forward(X, W1, b1, W2, b2)
pred_after, _ = forward(X, W1r, b1r, W2r, b2r)
print("reload max error", np.max(np.abs(pred_before - pred_after)))
assert np.allclose(pred_before, pred_after)

## XOR sanity
A one-hidden-layer ReLU network is capable of learning XOR.


In [ ]:
# XOR sanity check.
Xxor = np.array([[-1, -1], [-1, 1], [1, -1], [1, 1]], dtype=float)
yxor = np.array([0, 1, 1, 0])
gx = gc_rng.normal(0, 1, (2, 8))
gb = np.zeros(8)
gw = gc_rng.normal(0, 1, 8)
gbb = 0.0
for _ in range(3000):
    pp, hh = forward(Xxor, gx, gb, gw, gbb)
    dw1, db1, dw2, db2 = backward(Xxor, yxor, (pp, hh), gx, gb, gw, gbb)
    gx -= 0.1 * dw1
    gb -= 0.1 * db1
    gw -= 0.1 * dw2
    gbb -= 0.1 * db2
pred_xor, _ = forward(Xxor, gx, gb, gw, gbb)
print("XOR predictions", np.round(pred_xor, 3))
accuracy = np.mean((pred_xor >= 0.5) == yxor)
print("XOR accuracy", accuracy)
assert accuracy >= 0.75

## Failure analysis

A cold item with little prior history is hard to rank above established choices.

A popular item can receive a strong generic score even when it does not match the user's genre taste.

A sparse user history makes user-average and affinity estimates noisy.


## Results

```text
RESULTS
baseline:
chronological=0.118618, popularity=0.320436, Week 4 formula=0.121520, own Week 4 weights=0.117321, least-squares straight line=0.770856
solution:
HIDDEN=32, LEARNING_RATE=0.02, EPOCHS=80, val NDCG@10=0.779734, test NDCG@10=0.754571
runtime:
Training the selected model took about 63 seconds on this CPU; one forward pass is O(N(6H+H)) for N post-user pairs and H hidden units.
memory / model size:
Parameter count=6*32+32+32+1=257; like_model.npz plus normalization metadata.
3 failure cases:
A cold item with little prior history is hard to rank above established choices.
A popular item can receive a strong generic score even when it does not match the user's genre taste.
A sparse user history makes user-average and affinity estimates noisy.
what I would try next:
Try regularization and validation-based early stopping before adding more signals or layers.
```
